In [2]:
import os, json, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Cargar configuración
with open("../config.json") as f:
    config = json.load(f)

df = pd.read_csv(os.path.join("..", config["data"]["path"]))
target = config["data"]["target_column"]

# Feature derivado: antigüedad del vehículo
df["Vehicle_Age"] = 2025 - df["Year"]
df.drop(columns=["Year", "Car_Name"], inplace=True)

# Separar X e y
X = df.drop(columns=[target])
y = df[target]

# Clasificación de variables
categorical = X.select_dtypes(include="object").columns.tolist()
numerical = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Pipelines
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical),
    ("cat", cat_pipeline, categorical)
])

# Transformación
X_transformed = preprocessor.fit_transform(X)

# División
X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size=0.2, random_state=42)

# Al final de ft_engineering.py
def get_data():
    return X_train, X_test, y_train, y_test

